# Production observability with traces, spans, and replay

A multi-step LLM workflow in production fails about 5% of the time and nobody knows where. Errors surface as "the bot said something weird" support tickets. You have 75 minutes to add structured tracing with GenAI semantic attributes, ship traces to Langfuse Cloud, build a replay tool that re-runs a captured trace against a different model version, and find the planted bug in a `broken_pipeline` the lab ships.

The workflow simulates an order-status bot: classify intent, call a `get_order_status` tool, compose a response. The broken version returns the order price with a leading `$` from the tool, then the composer prefixes another `$` so the final output reads `$$45.99`. You will not see this from a glance at the logs; you see it in the trace.

**Duration:** about 75 minutes of work in a 105-minute session window.

**Cost preamble.** About a buck. Langfuse Cloud is free for this volume; the spend is Claude tokens (Sonnet for the workflow, Haiku for the cheap replay model).

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. langfuse 2.50+ supports the v2 SDK pattern
# (start_as_current_span, fetch_traces). httpx is the implicit transport.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 langfuse==2.50.0 httpx==0.27.2

In [ ]:
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import re
import sys
import time
import uuid
from datetime import datetime, timezone
from typing import Optional

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "production-observability-traces-spans-replay"
LAB_TAG_VALUE = LAB_ID

ROOT_CAUSE_PATH = "/content/root_cause.txt"
REPLAY_RESULTS_PATH = "/content/replay_results.json"

ANTHROPIC_SONNET = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

# Tag used to scope traces produced by this lab so cleanup can filter them.
TRACE_TAG = f"labrun_lab_slug:{LAB_ID}"

In [ ]:
# NBVAL_SKIP
# BYOK: session token, Anthropic key, Langfuse public+secret+host.
# Smoke-test each. Then register session.

import anthropic
from langfuse import Langfuse

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
LANGFUSE_PUBLIC_KEY = getpass.getpass("LANGFUSE_PUBLIC_KEY: ").strip()
LANGFUSE_SECRET_KEY = getpass.getpass("LANGFUSE_SECRET_KEY: ").strip()
LANGFUSE_HOST = input("LANGFUSE_HOST (press enter for https://cloud.langfuse.com): ").strip() or "https://cloud.langfuse.com"

if not all([ANTHROPIC_API_KEY, LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY]):
    print("Anthropic key + Langfuse keys are all required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)

try:
    # auth_check exists on the v2 SDK; if it does not, fall through to a cheap
    # fetch and let it raise if the keys are bad.
    if hasattr(langfuse, "auth_check"):
        langfuse.auth_check()
    print(f"Langfuse credentials validated against {LANGFUSE_HOST}.")
except Exception as exc:
    print(f"Langfuse keys rejected: {exc!r}")
    print("Double check that public goes in LANGFUSE_PUBLIC_KEY and secret in LANGFUSE_SECRET_KEY (easy to swap).")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"langfuse_host": LANGFUSE_HOST},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit safety net, orphan scan.
# Langfuse Cloud free tier does not offer a trace-delete API per project for
# external clients; the adapter tags the traces with the lab slug and the
# operator deletes the tagged set from the dashboard. The manifest reflects
# that with an explicit warning resource.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=REPLAY_RESULTS_PATH,
        region="local",
        cli_delete_command=f"rm -f {REPLAY_RESULTS_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=ROOT_CAUSE_PATH,
        region="local",
        cli_delete_command=f"rm -f {ROOT_CAUSE_PATH}",
    ),
    CleanupResource(
        type="langfuse_trace_set",
        id=f"tag={TRACE_TAG}",
        region="langfuse",
        cli_delete_command=(
            f"# Langfuse Cloud: open https://cloud.langfuse.com, filter traces by tag "
            f"{TRACE_TAG!r}, bulk delete from the dashboard."
        ),
    ),
]


class ObservabilityLabCleanupAdapter:
    """Local files + a soft cleanup for Langfuse trace tags.

    TODO: labrun-checks ships an observability adapter in 0.8+; until then the
    Langfuse cleanup path emits a warning and leaves manual deletion to the
    operator. Tagged traces auto-expire on the free tier after 30 days, so
    the soft path is acceptable for the lab.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "langfuse_trace_set":
            # The Langfuse v2 SDK does not expose a bulk-delete-by-tag for
            # client keys on the free tier. We flush any pending writes and
            # print a one-line note. Traces auto-expire in 30 days.
            try:
                langfuse.flush()
            except Exception:
                pass
            print(
                f"Langfuse trace set tagged {TRACE_TAG!r} flushed. "
                f"Free-tier projects do not offer programmatic bulk delete; "
                f"the traces auto-expire after 30 days, or delete via the dashboard."
            )

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "langfuse_trace_set":
            # We treat the soft cleanup as terminal: describe_resource returns
            # False so the verification scan accepts it.
            return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # No AWS tag store to scan; the Langfuse trace tag is handled above.
        return []


CLEANUP_ADAPTER = ObservabilityLabCleanupAdapter()


# Orphan scan: check the two local paths.
_orphans = [p for p in (REPLAY_RESULTS_PATH, ROOT_CAUSE_PATH) if os.path.exists(p)]
if _orphans:
    print("Orphan scan found leftover files from a prior session:")
    for p in _orphans:
        print(f"  {p}")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Workflow definition (read; do not edit)

The lab ships two versions of the order-status bot. `pipeline_good` works; `pipeline_broken` has the planted bug. Both have the same three steps: `classify_intent`, `get_order_status`, `compose_response`. You will instrument the working version in Task 1, then debug the broken one in Task 3.

In [ ]:
# Pipeline definitions. The broken tool returns the price as a string with a
# leading "$"; the composer adds another "$" so the final answer reads "$$45.99".

ORDER_DB = {
    "1001": {"price": 45.99, "status": "shipped"},
    "1002": {"price": 12.50, "status": "delivered"},
    "1003": {"price": 199.00, "status": "pending"},
}


def tool_get_order_status_good(order_id):
    """Working tool. Returns floats and strings cleanly."""
    rec = ORDER_DB[order_id]
    return {"price": rec["price"], "status": rec["status"]}


def tool_get_order_status_broken(order_id):
    """Broken tool. Returns price as a string with a literal '$' prefix."""
    rec = ORDER_DB[order_id]
    return {"price": f"${rec['price']:.2f}", "status": rec["status"]}


def llm_call(model, system, user, max_tokens=128):
    resp = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return resp


def classify_intent(question, model=ANTHROPIC_SONNET):
    resp = llm_call(
        model=model,
        system="Classify the user question into one of: order_status, return, other. Reply with the single word.",
        user=question,
        max_tokens=8,
    )
    return resp.content[0].text.strip().lower(), resp


def compose_response(question, tool_result, model=ANTHROPIC_SONNET):
    """Composer: builds a sentence using the tool output. The bug is that
    the composer prefixes 'Your order total is $' even when tool_result["price"]
    is itself a string with a leading dollar sign.
    """
    resp = llm_call(
        model=model,
        system=(
            "You are a friendly support agent. Compose a one-sentence reply that "
            "states the order status and total. Use the format: "
            "'Your order is <status>; your order total is $<price>.'"
        ),
        user=(
            f"Question: {question}\n"
            f"Order status: {tool_result['status']}\n"
            f"Order total: {tool_result['price']}"
        ),
        max_tokens=64,
    )
    return resp.content[0].text.strip(), resp


def pipeline_good(question, order_id, model=ANTHROPIC_SONNET):
    intent, _ = classify_intent(question, model)
    tool_result = tool_get_order_status_good(order_id)
    answer, _ = compose_response(question, tool_result, model)
    return {"intent": intent, "tool_result": tool_result, "answer": answer}


def pipeline_broken(question, order_id, model=ANTHROPIC_SONNET):
    intent, _ = classify_intent(question, model)
    tool_result = tool_get_order_status_broken(order_id)
    answer, _ = compose_response(question, tool_result, model)
    return {"intent": intent, "tool_result": tool_result, "answer": answer}


print("Workflow defined. Two pipelines: pipeline_good and pipeline_broken.")

## Task 1: Add Langfuse spans to the working pipeline

Wrap the workflow in Langfuse spans: a root trace for the workflow, child spans for `classify_intent`, `tool_get_order_status_good`, and `compose_response`. Each LLM-call span sets the GenAI semantic attributes:

- `gen_ai.system` = `"anthropic"`
- `gen_ai.request.model` = the model id
- `gen_ai.usage.input_tokens` and `gen_ai.usage.output_tokens` from `resp.usage`

Tag every trace with `{TRACE_TAG}` so cleanup can find them. Run the instrumented workflow against 10 sample questions.

In [ ]:
# Task 1: instrument pipeline_good with Langfuse spans.

SAMPLE_QUESTIONS = [
    ("What is the status of order 1001?", "1001"),
    ("Where is my order 1002?", "1002"),
    ("Has order 1003 shipped?", "1003"),
    ("Order 1001 update please.", "1001"),
    ("How is 1002 coming along?", "1002"),
    ("Status of 1003.", "1003"),
    ("Tracking for 1001.", "1001"),
    ("Order 1002 status?", "1002"),
    ("1003 delivery?", "1003"),
    ("Where's 1001 at?", "1001"),
]


def instrumented_pipeline_good(question, order_id, model=ANTHROPIC_SONNET, label="instrumented"):
    """Wrap pipeline_good with Langfuse traces + spans.

    On a successful run, return the same dict as pipeline_good plus a
    "trace_id" key.
    """
    trace = langfuse.trace(
        name="order_status_workflow",
        input={"question": question, "order_id": order_id},
        tags=[TRACE_TAG, label],
        metadata={"label": label, "model": model},
    )

    # YOUR CODE: create a span for "classify_intent" via trace.span(...).
    # Call classify_intent(question, model). Set the GenAI attributes via
    # span.update(metadata={"gen_ai.system": "anthropic", "gen_ai.request.model": model,
    #   "gen_ai.usage.input_tokens": resp.usage.input_tokens, "gen_ai.usage.output_tokens":
    #   resp.usage.output_tokens}). End the span.
    # YOUR CODE: create a span for "get_order_status" with the tool name, the order_id
    # as input, and the tool result as output. End the span.
    # YOUR CODE: create a span for "compose_response" with the same GenAI attributes
    # as classify_intent. End the span.
    # YOUR CODE: set trace.update(output={"answer": answer, "tool_result": tool_result}).
    # Return {"intent": intent, "tool_result": tool_result, "answer": answer,
    #         "trace_id": trace.id}.
    raise NotImplementedError("Replace with the spans + attribute calls.")


# Drive 10 traces.
trace_ids = []
for q, oid in SAMPLE_QUESTIONS:
    r = instrumented_pipeline_good(q, oid, label="task_1_clean")
    trace_ids.append(r["trace_id"])

langfuse.flush()
time.sleep(2)  # let traces propagate
print(f"Emitted {len(trace_ids)} traces. Sample id: {trace_ids[0]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: traces fetched from Langfuse include the expected span tree
# and the GenAI semantic attributes on at least one LLM call.


def checkpoint_1(session):
    try:
        # The v2 SDK exposes fetch_traces or get_traces depending on minor
        # version. Try both; one will work.
        if hasattr(langfuse, "fetch_traces"):
            traces = langfuse.fetch_traces(tags=[TRACE_TAG], limit=20).data
        elif hasattr(langfuse, "api") and hasattr(langfuse.api, "trace"):
            traces = langfuse.api.trace.list(tags=[TRACE_TAG], limit=20).data
        else:
            return CheckpointResult(
                status="error",
                error_reason="Langfuse SDK does not expose fetch_traces or api.trace.list; pin to langfuse==2.50.0.",
            )
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=f"fetch_traces failed: {exc!r}")

    if len(traces) < 1:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No traces tagged {TRACE_TAG!r}. The instrumented pipeline did not flush. "
                f"Check that trace.update(...) is called and langfuse.flush() runs."
            ),
        )
    # Confirm at least one trace has the expected span structure.
    sample_trace_id = traces[0].id
    try:
        if hasattr(langfuse, "fetch_trace"):
            detail = langfuse.fetch_trace(sample_trace_id).data
        else:
            detail = langfuse.api.trace.get(sample_trace_id)
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=f"fetch_trace failed: {exc!r}")

    obs = getattr(detail, "observations", None) or []
    span_names = {getattr(o, "name", None) for o in obs}
    expected = {"classify_intent", "compose_response", "get_order_status"}
    missing = expected - span_names
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Trace is missing expected span names: {missing}. Confirm each step calls "
                f"trace.span(name=...) and the spans are committed before flush."
            ),
        )
    # Check GenAI attributes on at least one LLM span.
    for o in obs:
        meta = getattr(o, "metadata", None) or {}
        if isinstance(meta, dict) and meta.get("gen_ai.system") == "anthropic":
            return CheckpointResult(status="pass")
    return CheckpointResult(
        status="fail",
        error_reason="No span has gen_ai.system=anthropic in metadata. Set the GenAI semantic attributes on the LLM-call spans.",
    )


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Langfuse v2 wants a `trace` then nested `span`s. Each span has its own start, optional metadata update, and end. The GenAI attributes live on `metadata`; Langfuse does not yet enforce the OTel semantic conventions, so the keys are convention-only ("gen_ai.system", "gen_ai.request.model", etc.).

</details>

<details><summary>Hint 2 (direction)</summary>

```
span = trace.span(name="classify_intent", input={"question": question})
intent, resp = classify_intent(question, model)
span.update(
    output={"intent": intent},
    metadata={
        "gen_ai.system": "anthropic",
        "gen_ai.request.model": model,
        "gen_ai.usage.input_tokens": resp.usage.input_tokens,
        "gen_ai.usage.output_tokens": resp.usage.output_tokens,
    },
)
span.end()
```

Same pattern for compose_response. The tool span has no GenAI metadata; just `input` (the order_id) and `output` (the tool dict).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def instrumented_pipeline_good(question, order_id, model=ANTHROPIC_SONNET, label="instrumented"):
    trace = langfuse.trace(
        name="order_status_workflow",
        input={"question": question, "order_id": order_id},
        tags=[TRACE_TAG, label],
        metadata={"label": label, "model": model},
    )

    s1 = trace.span(name="classify_intent", input={"question": question})
    intent, r1 = classify_intent(question, model)
    s1.update(
        output={"intent": intent},
        metadata={
            "gen_ai.system": "anthropic",
            "gen_ai.request.model": model,
            "gen_ai.usage.input_tokens": r1.usage.input_tokens,
            "gen_ai.usage.output_tokens": r1.usage.output_tokens,
        },
    )
    s1.end()

    s2 = trace.span(name="get_order_status", input={"order_id": order_id})
    tool_result = tool_get_order_status_good(order_id)
    s2.update(output=tool_result)
    s2.end()

    s3 = trace.span(name="compose_response", input={"question": question, "tool_result": tool_result})
    answer, r3 = compose_response(question, tool_result, model)
    s3.update(
        output={"answer": answer},
        metadata={
            "gen_ai.system": "anthropic",
            "gen_ai.request.model": model,
            "gen_ai.usage.input_tokens": r3.usage.input_tokens,
            "gen_ai.usage.output_tokens": r3.usage.output_tokens,
        },
    )
    s3.end()

    trace.update(output={"answer": answer, "tool_result": tool_result})
    return {"intent": intent, "tool_result": tool_result, "answer": answer, "trace_id": trace.id}
```

</details>

**Wallet check.** 10 traces, three Sonnet calls each (30 calls total, ~80 input tokens / 30 output tokens each at $3/$15 per 1M): roughly 25 cents. Langfuse storage is free. Cumulative spend: ~25 cents.

## Task 2: Build the replay tool

`replay(trace_id, model)` reads the original trace's question + order_id, calls the instrumented pipeline again with a different model override, and stamps the new trace with `replay_of=<original_trace_id>` in its tags so the link is queryable.

Run replay against one of the Task 1 traces with `model=ANTHROPIC_HAIKU`. Confirm the new trace exists in Langfuse.

In [ ]:
# Task 2: replay tool.

def fetch_trace(trace_id):
    if hasattr(langfuse, "fetch_trace"):
        return langfuse.fetch_trace(trace_id).data
    return langfuse.api.trace.get(trace_id)


def replay(trace_id, model):
    # YOUR CODE: fetch the original trace via fetch_trace(trace_id).
    # Extract original_question and original_order_id from the trace's input
    # field (a dict; if it is a string, json.loads it first).
    # YOUR CODE: call instrumented_pipeline_good with the same question + order_id
    # but the new model; pass label=f"replay_of:{trace_id}" so the new trace's
    # tag set carries the link.
    # Return the result dict (includes the new trace_id).
    raise NotImplementedError("Replace with the fetch + re-run.")


original_trace_id = trace_ids[0]
replay_result = replay(original_trace_id, ANTHROPIC_HAIKU)
print(f"Replay produced new trace: {replay_result['trace_id']}")
print(f"Replay link tag on the new trace: replay_of:{original_trace_id}")
langfuse.flush()
time.sleep(2)

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the replayed trace exists in Langfuse and carries the
# replay_of link in its tags.


def checkpoint_2(session):
    new_trace_id = replay_result["trace_id"]
    try:
        if hasattr(langfuse, "fetch_trace"):
            detail = langfuse.fetch_trace(new_trace_id).data
        else:
            detail = langfuse.api.trace.get(new_trace_id)
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=f"fetch_trace failed: {exc!r}")
    tags = list(getattr(detail, "tags", []) or [])
    expected_tag = f"replay_of:{original_trace_id}"
    if expected_tag not in tags:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"New trace {new_trace_id} does not carry tag {expected_tag!r}. "
                f"Tags present: {tags!r}. Confirm the replay passes label=f'replay_of:{{trace_id}}'."
            ),
        )
    # Confirm a different model was actually used.
    meta = getattr(detail, "metadata", None) or {}
    if isinstance(meta, dict) and meta.get("model") == ANTHROPIC_SONNET:
        return CheckpointResult(
            status="fail",
            error_reason="Replay used Sonnet (the original). Pass model=ANTHROPIC_HAIKU to replay.",
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

`replay` is two moves: fetch the source trace, re-invoke the pipeline with a model override. The link between the new and original trace lives in the new trace's tag set; that is how downstream tooling correlates the pair.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def replay(trace_id, model):
    detail = fetch_trace(trace_id)
    src_in = detail.input
    if isinstance(src_in, str):
        src_in = json.loads(src_in)
    return instrumented_pipeline_good(
        src_in["question"],
        src_in["order_id"],
        model=model,
        label=f"replay_of:{trace_id}",
    )
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2. The only edge case: if `detail.input` is a JSON string (some Langfuse SDK versions serialize on the way out), call `json.loads` on it first.

</details>

**Wallet check.** One replay run: three Haiku calls at $0.80/$4 per 1M. Pennies. Cumulative: ~25 cents.

## Task 3: Find the bug in the broken pipeline

Run `pipeline_broken` 5 times via an instrumented wrapper. The outputs will read like `"$$45.99"` or similar. Inspect the trace's tool span; the bug is visible there but not in the user-facing answer alone.

Write a one-paragraph root-cause description to `/content/root_cause.txt`. The validator regex-matches against the expected description. The regex is lenient; you only need to name the type-mismatch in the tool output and the double-prefix in the composer.

In [ ]:
# Task 3: instrument + run pipeline_broken, write root cause.

def instrumented_pipeline_broken(question, order_id, model=ANTHROPIC_SONNET, label="task_3_broken"):
    trace = langfuse.trace(
        name="order_status_workflow",
        input={"question": question, "order_id": order_id},
        tags=[TRACE_TAG, label],
        metadata={"label": label, "model": model},
    )
    s1 = trace.span(name="classify_intent", input={"question": question})
    intent, r1 = classify_intent(question, model)
    s1.update(output={"intent": intent})
    s1.end()
    s2 = trace.span(name="get_order_status", input={"order_id": order_id})
    tool_result = tool_get_order_status_broken(order_id)
    s2.update(output=tool_result)
    s2.end()
    s3 = trace.span(name="compose_response", input={"question": question, "tool_result": tool_result})
    answer, _ = compose_response(question, tool_result, model)
    s3.update(output={"answer": answer})
    s3.end()
    trace.update(output={"answer": answer})
    return {"answer": answer, "tool_result": tool_result, "trace_id": trace.id}


broken_results = []
for q, oid in SAMPLE_QUESTIONS[:5]:
    r = instrumented_pipeline_broken(q, oid)
    print(f"  answer: {r['answer'][:120]}")
    broken_results.append(r)

langfuse.flush()
time.sleep(2)

# Inspect the first broken trace's tool span via the SDK so the operator can
# read it without flipping to the dashboard.
broken_trace_detail = fetch_trace(broken_results[0]["trace_id"])
for o in (getattr(broken_trace_detail, "observations", None) or []):
    if getattr(o, "name", None) == "get_order_status":
        print(f"Tool span output captured in trace: {getattr(o, 'output', None)!r}")
        break

# YOUR CODE: write a one-paragraph root-cause description to ROOT_CAUSE_PATH.
# The bug is: tool_get_order_status_broken returns price as a string with a
# leading dollar sign, and compose_response prefixes another dollar sign in
# its prompt template, producing $$XX.XX in the final answer.
# Use words like "string", "type", "dollar", "double", "$"; the validator
# matches a lenient regex over these.

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: root_cause.txt exists and matches the lenient regex.


def checkpoint_3(session):
    if not os.path.exists(ROOT_CAUSE_PATH):
        return CheckpointResult(status="fail", error_reason=f"{ROOT_CAUSE_PATH} missing.")
    body = open(ROOT_CAUSE_PATH).read().lower()
    must_mention = [
        re.compile(r"\bstring\b|\btype\b"),
        re.compile(r"dollar|\$"),
        re.compile(r"\bdouble\b|twice|prefix|already|leading"),
    ]
    for pat in must_mention:
        if not pat.search(body):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Root cause description does not mention the expected concept (matched by {pat.pattern!r}). "
                    f"Describe: the tool returns price as a STRING with a leading $; the composer adds ANOTHER $; "
                    f"so the final output shows $$XX.XX."
                ),
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The bug is not in the LLM. It is in the data passed between the tool and the composer. Read the tool span's output value in the trace; you will see the type mismatch.

</details>

<details><summary>Hint 2 (direction)</summary>

`tool_get_order_status_broken` returns `{"price": "$45.99", "status": "shipped"}` (price is a string with a literal "$" prefix). The composer prompt template hard-codes "Your order total is $<price>." When `<price>` is already "$45.99", the final sentence reads "Your order total is $$45.99." Fix it in either layer; both fixes are reasonable.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
with open(ROOT_CAUSE_PATH, "w") as f:
    f.write(
        "The tool returns price as a string with a leading dollar sign ('$45.99'). "
        "The composer prompt template prefixes another '$' (Your order total is $<price>). "
        "The combined output shows the dollar sign twice ('$$45.99'). "
        "Fix in the tool by returning a float, or fix in the composer by NOT prefixing '$' "
        "when the price already contains it. Cleanest fix: tool returns floats only."
    )
```

</details>

**Wallet check.** 5 broken-pipeline runs at Sonnet rates: ~$0.15. Cumulative: ~$0.40.

## Task 4: Replay the broken trace against the FIXED pipeline

Apply the fix (we choose: fix in the tool, return floats). Re-run the same questions through the now-fixed `instrumented_pipeline_good`, then write the per-question (broken_answer, fixed_answer, status) tuples to `/content/replay_results.json`. The fixed_answer must NOT contain `$$`.

In [ ]:
# Task 4: apply the fix (the lab fixes the tool; the composer is unchanged).

# YOUR CODE: assign the fixed pipeline. The simplest fix is to use
# instrumented_pipeline_good as-is (it already calls tool_get_order_status_good
# which returns floats). So `replay_results` below should call the good
# instrumented pipeline for each question.

replay_results = []
for (q, oid), broken in zip(SAMPLE_QUESTIONS[:5], broken_results):
    # YOUR CODE: call instrumented_pipeline_good(q, oid, label="task_4_fixed")
    # and append a dict with question, order_id, broken_answer (broken["answer"]),
    # fixed_answer (the new run's answer), and status="ok" to replay_results.
    raise NotImplementedError("Replace with the fixed-pipeline call.")

with open(REPLAY_RESULTS_PATH, "w") as f:
    json.dump(replay_results, f, indent=1)

print(f"Wrote {len(replay_results)} replay rows to {REPLAY_RESULTS_PATH}")
langfuse.flush()

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: fixed answers do not contain "$$".


def checkpoint_4(session):
    if not os.path.exists(REPLAY_RESULTS_PATH):
        return CheckpointResult(status="fail", error_reason=f"{REPLAY_RESULTS_PATH} missing.")
    rows = json.loads(open(REPLAY_RESULTS_PATH).read())
    if len(rows) < 5:
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected 5 rows in replay_results, found {len(rows)}.",
        )
    for r in rows:
        if "$$" in r.get("fixed_answer", ""):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Fixed answer for order_id={r.get('order_id')} still has '$$': {r['fixed_answer']!r}. "
                    f"The fix was not applied; instrumented_pipeline_good calls the GOOD tool."
                ),
            )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

You already have the fixed pipeline: `instrumented_pipeline_good` calls `tool_get_order_status_good` which returns floats. The fix is to call the good pipeline for the replay, not the broken one.

</details>

<details><summary>Hint 2 (direction)</summary>

```
for (q, oid), broken in zip(SAMPLE_QUESTIONS[:5], broken_results):
    fixed = instrumented_pipeline_good(q, oid, label="task_4_fixed")
    replay_results.append({
        "question": q,
        "order_id": oid,
        "broken_answer": broken["answer"],
        "fixed_answer": fixed["answer"],
        "status": "ok",
    })
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Five fixed-pipeline runs at Sonnet rates: another ~$0.15. Cumulative: ~$0.55.

## Cleanup

Local files first. The Langfuse traces flush on cleanup; the free tier does not expose a programmatic bulk-delete by tag, so the cleanup output names the tag for manual deletion from the dashboard (or the traces auto-expire after 30 days). Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, the trace deletion was not requested. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.60.** Sonnet dominated; Haiku was pennies on the replay; Langfuse Cloud was free. Local files cleared; tagged traces flushed (auto-expire on the free tier).

## Reflection

These are not graded. They are for you.

1. Your replay tool re-runs against a different model. The team wants to replay against the SAME model but a different prompt version (the system prompt changed). What is the minimal change to your replay function, and what does the new trace's tag set look like?

2. The broken pipeline shipped to production for 3 days before someone noticed. What single observability signal would have caught it sooner (something that the current GenAI semantic attributes do not capture)?

## Exam-Style Practice

**Question 1 (span attributes for debugging):**

A multi-step LLM workflow produces nonsense output. The team adds OTel-style spans. Which attribute most reliably surfaces the root cause when the model output looks normal but is semantically wrong?

A. `gen_ai.usage.input_tokens`
B. `gen_ai.response.finish_reason`
C. `gen_ai.system`
D. `gen_ai.request.temperature`

<details><summary>Show answer</summary>

**Correct: B.**

- A surfaces cost, not correctness.
- B is correct: `finish_reason` distinguishes "length" (cut off mid-thought), "stop" (clean stop), "tool_calls" (routed to a tool). "Length" on a step that should have produced a full sentence is the canonical signal that the prompt is wrong or max_tokens is too low.
- C identifies the provider; not a debugging signal.
- D is a generation parameter; it does not surface failure.

</details>

**Question 2 (replay vs eval):**

A team has an eval harness that scores model outputs against a golden set. They also have a replay tool that re-runs a captured trace against a different model. Which question does replay answer that eval does not?

A. Which model produces the highest aggregate score on the golden set?
B. How would the production traffic of last week have looked under the new model?
C. Which prompts in the golden set are hardest?
D. Which model has the lowest cost per query?

<details><summary>Show answer</summary>

**Correct: B.**

- A is the eval harness's primary job.
- B is correct: replay answers the counterfactual on real production traffic, which eval cannot because eval runs against a curated golden set, not production distribution.
- C is the eval harness's secondary job (slicing by difficulty).
- D is the cost-attribution surface, not replay.

</details>

**Question 3 (trace volume cost):**

A team logs traces for every request on a 10K-QPS workload, six spans per request. The observability bill jumps. Which is the highest-leverage first change?

A. Sample traces at 10%.
B. Drop the GenAI semantic attributes from the spans.
C. Move to a cheaper observability vendor.
D. Reduce the span tree from six spans to one.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: sampling at 10% drops 90% of trace volume in one config change; head sampling preserves rare error traces if combined with tail-based exception sampling.
- B is wrong: the attributes are tiny; the volume is in the trace count, not the per-span payload size.
- C is a real lever but moves the problem rather than fixing it.
- D is wrong: the spans are the debugging signal; flattening to one removes the per-step inspection that just paid off in Task 3.

</details>